In [1]:
import os
import sys
import pickle
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Pipfile").exists()),
    Path("../../..").resolve(),
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from load_data.load_conformal_data import load_suffix_results
from conformance_checking.conformance import ConformanceChecking
from conformance_checking.conformance_results import ConformanceResults

print("Project root:", PROJECT_ROOT)


Project root: /Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/Conformance Checking Mustroph


In [2]:
# Load model and suffix validation set and determine output directory:

# Helpdesk
log_name = "Helpdesk"
path_to_be_model = PROJECT_ROOT / "data" / "process_models" / "Helpdesk.bpmn"
path_calibration_samples_results = PROJECT_ROOT / "data" / "Helpdesk" / "proact_conf_check_v2" / "eval_validation_set"
output_dir = PROJECT_ROOT / "data" / "Helpdesk" / "proact_conf_check_v2" / "offline_analysis"

# Sepsis:
# log_name = "Sepsis"
# path_to_be_model = PROJECT_ROOT / "data" / "process_models" / "Sepsis.bpmn"
# path_calibration_samples_results = PROJECT_ROOT / "data" / "Sepsis" / "proact_conf_check_v2" / "eval_validation_set"
# output_dir = PROJECT_ROOT / "data" / "Sepsis" / "proact_conf_check_v2" / "offline_analysis"

# BPIC20:
# log_name = "BPIC20-DD"
# path_to_be_model = PROJECT_ROOT / "data" / "process_models" / "BPIC20_DD.bpmn"
# path_calibration_samples_results = PROJECT_ROOT / "data" / "BPIC20" / "proact_conf_check_v2" / "eval_validation_set"
# output_dir = PROJECT_ROOT / "data" / "BPIC20" / "proact_conf_check_v2" / "offline_analysis"

# Repair
# log_name = "Repair"
# path_to_be_model = PROJECT_ROOT / "data" / "process_models" / "Repair.bpmn"
# path_calibration_samples_results = PROJECT_ROOT / "data" / "repair_shop" / "proact_conf_check_v2" / "eval_validation_set"
# output_dir = PROJECT_ROOT / "data" / "repair_shop" / "proact_conf_check_v2" / "offline_analysis"

print("Process model:", path_to_be_model)
print("Suffix samples:", path_calibration_samples_results)
print("Output:", output_dir)
assert path_to_be_model.exists(), f"Process model not found: {path_to_be_model}"
assert path_calibration_samples_results.exists(), (
    f"Suffix samples not found: {path_calibration_samples_results}"
)


Process model: /Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/Conformance Checking Mustroph/data/process_models/Helpdesk.bpmn
Suffix samples: /Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/Conformance Checking Mustroph/data/Helpdesk/proact_conf_check_v2/eval_validation_set
Output: /Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/Conformance Checking Mustroph/data/Helpdesk/proact_conf_check_v2/offline_analysis


In [3]:
# MC Suffix Sampling Results
# results (case_id, prefix_len)
results = load_suffix_results(path=path_calibration_samples_results)
print(results[0].keys())

n_prefixes = sum(len(chunk) for chunk in results)
print(f"Loaded {len(results)} file(s) with {n_prefixes} prefixes")

# Conformance Checking object:
# Hier ist das Modell vorhanden, das durch den Inductive Miner erstellt wurde
cc = ConformanceChecking(path_process_model=str(path_to_be_model))

# Evaluate the probabilistic suffix samples using conformance checking:
cfr = ConformanceResults(log_name=log_name, data=results, conformance_object=cc)
workers = min(8, os.cpu_count() or 1)
calibration_conformance_dict = cfr.conformance_results(target_workers=workers)
print("Finished conformance checking computation")

# Testing purpose
# test = cfr.quick_fitness()

os.makedirs(output_dir, exist_ok=True)

# Batch size to store multiple tuples
n = 50
# Dicitionary ist 
# calibration_conformance_dict = {
#     "case_id": [...],
#     "target_conformance": [...],
#     "ml_conformance": [...],
#     "samples_conformance": [...]
# }
num_cases = len(calibration_conformance_dict['case_id'])
total_saved = 0
for i in range(0, num_cases, n):
    # Erster Batch nimmt die ersten 50 einträge  aus allenListen
    #     batch = {
    #     "case_id": case_id[0:50],
    #     "target_conformance": target_conformance[0:50],
    #     "ml_conformance": ml_conformance[0:50],
    #     "samples_conformance": samples_conformance[0:50]
    # }
    batch = {k: v[i:i+n] for k, v in calibration_conformance_dict.items()}

    chunk_number = (i // n + 1) * n
    filename = os.path.join(output_dir, f"conformance_res_{chunk_number:03d}.pkl")

    with open(filename, "wb") as f:
        pickle.dump(batch, f)
    batch_size = len(batch["case_id"])
    total_saved += batch_size
    print(f"Saved {batch_size} entries to {filename}")

print(f"Saved a total of {total_saved} entries across {(num_cases + n - 1) // n} files")
print("All files stored!")


Looking for .pkl files in: /Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/Conformance Checking Mustroph/data/Helpdesk/proact_conf_check_v2/eval_validation_set
Resolved absolute path: /Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/Conformance Checking Mustroph/data/Helpdesk/proact_conf_check_v2/eval_validation_set
Found 1 files
Loaded all suffix results!
dict_keys([('Case 1', 1), ('Case 1', 2), ('Case 1', 3), ('Case 1', 4), ('Case 1001', 1), ('Case 1001', 2), ('Case 1001', 3), ('Case 1003', 1), ('Case 1003', 2), ('Case 1003', 3), ('Case 1007', 1), ('Case 1007', 2), ('Case 1007', 3), ('Case 1008', 1), ('Case 1008', 2), ('Case 1008', 3), ('Case 1011', 1), ('Case 1011', 2), ('Case 1011', 3), ('Case 1011', 4)])
Loaded 1 file(s) with 20 prefixes


aligning log, completed variants ::   0%|          | 0/5 [00:00<?, ?it/s]

aligning log, completed variants ::   0%|          | 0/5 [00:00<?, ?it/s]

aligning log, completed variants :: 100%|██████████| 5/5 [00:00<00:00, 80.86it/s]


aligning log, completed variants :: 100%|██████████| 7/7 [00:00<00:00, 93.60it/s]


Finished conformance checking computation
Saved 20 entries to /Users/laurensohl/Downloads/Gitarre/Akademia/Master_Arbeit/Conformance Checking Mustroph/data/Helpdesk/proact_conf_check_v2/offline_analysis/conformance_res_050.pkl
Saved a total of 20 entries across 1 files
All files stored!
